In [28]:
%pip install pandas numpy scikit-learn xgboost torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# ─── Imports & Configuration (baseline-aligned) ────────────────────────────
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
NUM_CLASSES = len(LABEL_COLS)
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.15

EMB_DIR_CANDIDATES = ["./final_embeddings", "./Embeddings", "./embeddings"]
EMB_DIR = next((d for d in EMB_DIR_CANDIDATES if os.path.isdir(d)), None)
if EMB_DIR is None:
    raise FileNotFoundError(f"No embeddings directory found in {EMB_DIR_CANDIDATES}")

RDKIT_FILE = "rdkit_descriptors.csv"
MOL2VEC_FILE = "mol2vec.csv"

np.random.seed(SEED)

print("✅ Imports loaded")
print(f"Embeddings dir: {EMB_DIR}")
print(f"Files: {RDKIT_FILE}, {MOL2VEC_FILE}")
print(f"Seed={SEED}, Test={TEST_SIZE}, Val={VAL_SIZE}")

✅ Imports loaded
Embeddings dir: ./final_embeddings
Files: rdkit_descriptors.csv, mol2vec.csv
Seed=42, Test=0.2, Val=0.15


In [30]:
# ─── Load RDKit + Mol2Vec ───────────────────────────────────────────────────
rdkit_path = os.path.join(EMB_DIR, RDKIT_FILE)
mol2vec_path = os.path.join(EMB_DIR, MOL2VEC_FILE)

df_rdkit = pd.read_csv(rdkit_path)
df_mol2vec = pd.read_csv(mol2vec_path)

labels_rdkit = df_rdkit[LABEL_COLS].values.argmax(axis=1)
labels_mol2vec = df_mol2vec[LABEL_COLS].values.argmax(axis=1)

if len(df_rdkit) != len(df_mol2vec):
    raise ValueError(f"Row count mismatch: RDKit={len(df_rdkit)}, Mol2Vec={len(df_mol2vec)}")
if not np.array_equal(labels_rdkit, labels_mol2vec):
    raise ValueError("Label mismatch between RDKit and Mol2Vec files. Ensure aligned rows.")

y = labels_rdkit

rdkit_feat_cols = [c for c in df_rdkit.columns if c not in LABEL_COLS]
mol2vec_feat_cols = [c for c in df_mol2vec.columns if c not in LABEL_COLS]

X_rdkit = df_rdkit[rdkit_feat_cols].values
X_mol2vec = df_mol2vec[mol2vec_feat_cols].values

print(f"RDKit loaded:   X={X_rdkit.shape}, y={y.shape}")
print(f"Mol2Vec loaded: X={X_mol2vec.shape}, y={y.shape}")
print(pd.Series(y).map(dict(enumerate(LABEL_COLS))).value_counts())

RDKit loaded:   X=(14716, 213), y=(14716,)
Mol2Vec loaded: X=(14716, 300), y=(14716,)
Sweet        9422
Undefined    2053
Sour         1534
Bitter       1515
Umami         192
Name: count, dtype: int64


In [31]:
# ─── Metrics (same style as baseline) ───────────────────────────────────────
def macro_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    specificities = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        spec_i = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec_i)
    return float(np.mean(specificities))


def compute_metrics(y_true, y_pred):
    return {
        "ACC": accuracy_score(y_true, y_pred),
        "PRE": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "SPEC": macro_specificity(y_true, y_pred),
        "SENS": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

print("✅ Metric helpers ready")

✅ Metric helpers ready


In [32]:
# ─── Split data (shared indices for both embeddings) ───────────────────────
indices = np.arange(len(y))

idx_train, idx_test, y_train, y_test = train_test_split(
    indices, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

idx_fit, idx_val, y_fit, y_val = train_test_split(
    idx_train, y_train, test_size=VAL_SIZE, random_state=SEED, stratify=y_train
)

# RDKit splits
X_fit_rdkit = X_rdkit[idx_fit]
X_val_rdkit = X_rdkit[idx_val]
X_test_rdkit = X_rdkit[idx_test]

# Mol2Vec splits
X_fit_mol2vec = X_mol2vec[idx_fit]
X_val_mol2vec = X_mol2vec[idx_val]
X_test_mol2vec = X_mol2vec[idx_test]

print(f"Fit={len(idx_fit)}, Val={len(idx_val)}, Test={len(idx_test)}")
print(f"RDKit fit/val/test: {X_fit_rdkit.shape}, {X_val_rdkit.shape}, {X_test_rdkit.shape}")
print(f"Mol2Vec fit/val/test: {X_fit_mol2vec.shape}, {X_val_mol2vec.shape}, {X_test_mol2vec.shape}")

Fit=10006, Val=1766, Test=2944
RDKit fit/val/test: (10006, 213), (1766, 213), (2944, 213)
Mol2Vec fit/val/test: (10006, 300), (1766, 300), (2944, 300)


In [33]:
# ─── Model config reference (baseline-aligned) ─────────────────────────────
# Extra Trees: n_estimators=300, random_state=SEED, n_jobs=-1
# XGBoost: n_estimators=300, learning_rate=0.1, max_depth=6,
#          eval_metric='mlogloss', use_label_encoder=False
# SVM: kernel='rbf', decision_function_shape='ovr', probability=True
print("✅ Baseline-aligned model configs ready")

✅ Baseline-aligned model configs ready


In [34]:
# ─── Train base models requested by user ───────────────────────────────────
# Models:
# 1) Extra Trees (RDKit)
# 2) XGBoost (RDKit)
# 3) SVM (RDKit)
# 4) XGBoost (Mol2Vec)

base_outputs = {}
model_embeddings = {}

# ---------- Extra Trees (RDKit) ----------
et_rdkit = ExtraTreesClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
)
et_rdkit.fit(X_fit_rdkit, y_fit)
base_outputs["Extra Trees (RDKit)"] = {
    "fit_proba": et_rdkit.predict_proba(X_fit_rdkit),
    "val_proba": et_rdkit.predict_proba(X_val_rdkit),
    "test_proba": et_rdkit.predict_proba(X_test_rdkit),
}
model_embeddings["Extra Trees (RDKit)"] = "RDKit"

# ---------- XGBoost (RDKit) ----------
xgb_rdkit = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="mlogloss",
    use_label_encoder=False,
)
xgb_rdkit.fit(X_fit_rdkit, y_fit)
base_outputs["XGBoost (RDKit)"] = {
    "fit_proba": xgb_rdkit.predict_proba(X_fit_rdkit),
    "val_proba": xgb_rdkit.predict_proba(X_val_rdkit),
    "test_proba": xgb_rdkit.predict_proba(X_test_rdkit),
}
model_embeddings["XGBoost (RDKit)"] = "RDKit"

# ---------- SVM (RDKit) ----------
svm_scaler_rdkit = StandardScaler()
X_fit_rdkit_s = svm_scaler_rdkit.fit_transform(X_fit_rdkit)
X_val_rdkit_s = svm_scaler_rdkit.transform(X_val_rdkit)
X_test_rdkit_s = svm_scaler_rdkit.transform(X_test_rdkit)

svm_rdkit = SVC(
    kernel="rbf",
    random_state=SEED,
    decision_function_shape="ovr",
    probability=True,
)
svm_rdkit.fit(X_fit_rdkit_s, y_fit)
base_outputs["SVM (RDKit)"] = {
    "fit_proba": svm_rdkit.predict_proba(X_fit_rdkit_s),
    "val_proba": svm_rdkit.predict_proba(X_val_rdkit_s),
    "test_proba": svm_rdkit.predict_proba(X_test_rdkit_s),
}
model_embeddings["SVM (RDKit)"] = "RDKit"

# ---------- XGBoost (Mol2Vec) ----------
xgb_mol2vec = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="mlogloss",
    use_label_encoder=False,
)
xgb_mol2vec.fit(X_fit_mol2vec, y_fit)
base_outputs["XGBoost (Mol2Vec)"] = {
    "fit_proba": xgb_mol2vec.predict_proba(X_fit_mol2vec),
    "val_proba": xgb_mol2vec.predict_proba(X_val_mol2vec),
    "test_proba": xgb_mol2vec.predict_proba(X_test_mol2vec),
}
model_embeddings["XGBoost (Mol2Vec)"] = "Mol2Vec"

print("✅ Requested base models trained:")
for n in base_outputs:
    print(f"  - {n}")

KeyboardInterrupt: 

In [ ]:
# ─── Ensemble by validation-weighted probability averaging ─────────────────
def pred_from_proba(proba):
    return np.argmax(proba, axis=1)

val_f1_scores = {}
for name, out in base_outputs.items():
    val_pred = pred_from_proba(out["val_proba"])
    val_f1_scores[name] = f1_score(y_val, val_pred, average="macro", zero_division=0)

score_sum = sum(val_f1_scores.values())
weights = {name: (score / score_sum if score_sum > 0 else 1/len(val_f1_scores)) for name, score in val_f1_scores.items()}

fit_ens_proba = sum(weights[name] * base_outputs[name]["fit_proba"] for name in base_outputs)
val_ens_proba = sum(weights[name] * base_outputs[name]["val_proba"] for name in base_outputs)
test_ens_proba = sum(weights[name] * base_outputs[name]["test_proba"] for name in base_outputs)

ensemble_outputs = {
    "fit_pred": pred_from_proba(fit_ens_proba),
    "val_pred": pred_from_proba(val_ens_proba),
    "test_pred": pred_from_proba(test_ens_proba),
}

print("✅ Ensemble ready")
print("Validation F1 weights:")
for name, w in weights.items():
    print(f"  {name}: {w:.4f} (val_F1={val_f1_scores[name]:.4f})")

✅ Ensemble ready
Validation F1 weights:
  Extra Trees (RDKit): 0.2538 (val_F1=0.8324)
  XGBoost (RDKit): 0.2568 (val_F1=0.8421)
  SVM (RDKit): 0.2424 (val_F1=0.7951)
  XGBoost (Mol2Vec): 0.2470 (val_F1=0.8100)


In [ ]:
# ─── Results table (base + ensemble) ───────────────────────────────────────
rows = []

for name, out in base_outputs.items():
    fit_pred = np.argmax(out["fit_proba"], axis=1)
    val_pred = np.argmax(out["val_proba"], axis=1)
    test_pred = np.argmax(out["test_proba"], axis=1)

    fit_metrics = compute_metrics(y_fit, fit_pred)
    val_metrics = compute_metrics(y_val, val_pred)
    test_metrics = compute_metrics(y_test, test_pred)

    row = {"Model": name, "Embedding": model_embeddings.get(name, "Mixed"), "Type": "Base"}
    for k, v in fit_metrics.items():
        row[f"Fit_{k}"] = round(v, 4)
    for k, v in val_metrics.items():
        row[f"Val_{k}"] = round(v, 4)
    for k, v in test_metrics.items():
        row[f"Test_{k}"] = round(v, 4)
    row["Ensemble_Weight"] = round(weights.get(name, np.nan), 4)
    rows.append(row)

fit_metrics_ens = compute_metrics(y_fit, ensemble_outputs["fit_pred"])
val_metrics_ens = compute_metrics(y_val, ensemble_outputs["val_pred"])
test_metrics_ens = compute_metrics(y_test, ensemble_outputs["test_pred"])

ens_row = {
    "Model": "Ensemble(ExtraTrees-RDKit + XGB-RDKit + SVM-RDKit + XGB-Mol2Vec)",
    "Embedding": "Mixed",
    "Type": "Ensemble",
    "Ensemble_Weight": np.nan,
}
for k, v in fit_metrics_ens.items():
    ens_row[f"Fit_{k}"] = round(v, 4)
for k, v in val_metrics_ens.items():
    ens_row[f"Val_{k}"] = round(v, 4)
for k, v in test_metrics_ens.items():
    ens_row[f"Test_{k}"] = round(v, 4)
rows.append(ens_row)

results_df = pd.DataFrame(rows)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 300)
pd.set_option("display.float_format", "{:.4f}".format)

print("✅ Results table ready")
results_df.sort_values(by="Test_F1", ascending=False).reset_index(drop=True)

✅ Results table ready


,Model,Embedding,Type,Fit_ACC,Fit_PRE,Fit_SPEC,Fit_SENS,Fit_F1,Fit_MCC,Val_ACC,Val_PRE,Val_SPEC,Val_SENS,Val_F1,Val_MCC,Test_ACC,Test_PRE,Test_SPEC,Test_SENS,Test_F1,Test_MCC,Ensemble_Weight
0,Ensemble(ExtraTrees-RDKit + XGB-RDKit + SVM-RD...,Mixed,Ensemble,0.9994,0.9991,0.9998,0.9991,0.9991,0.9989,0.8811,0.8495,0.9567,0.8219,0.8334,0.7814,0.8896,0.8685,0.9601,0.8422,0.8533,0.7971,NaN
1,XGBoost (RDKit),RDKit,Base,0.9994,0.9989,0.9999,0.9992,0.9991,0.9989,0.8851,0.8577,0.9574,0.8296,0.8421,0.7884,0.8835,0.8624,0.9594,0.8420,0.8509,0.7868,0.2568
2,Extra Trees (RDKit),RDKit,Base,0.9994,0.9990,0.9998,0.9992,0.9991,0.9989,0.8794,0.8516,0.9562,0.8172,0.8324,0.7777,0.8815,0.8499,0.9582,0.8304,0.8389,0.7829,0.2538
3,SVM (RDKit),RDKit,Base,0.8955,0.8677,0.9638,0.8382,0.8505,0.8099,0.8630,0.8069,0.9529,0.7869,0.7951,0.7499,0.8764,0.8287,0.9580,0.8121,0.8187,0.7746,0.2424
4,XGBoost (Mol2Vec),Mol2Vec,Base,0.9988,0.9972,0.9996,0.9968,0.9970,0.9978,0.8669,0.8248,0.9507,0.7989,0.8100,0.7538,0.8736,0.8321,0.9539,0.8081,0.8184,0.7665,0.2470


In [ ]:
# ─── Save outputs ───────────────────────────────────────────────────────────
results_df.to_csv("results_ensemble_rdkit_mol2vec_4models.csv", index=False)
weights_df = pd.DataFrame([
    {"Model": k, "Val_F1": v, "Weight": weights[k]} for k, v in val_f1_scores.items()
])
weights_df.to_csv("ensemble_weights_rdkit_mol2vec_4models.csv", index=False)

print("💾 Saved results_ensemble_rdkit_mol2vec_4models.csv")
print("💾 Saved ensemble_weights_rdkit_mol2vec_4models.csv")

💾 Saved results_ensemble_rdkit_mol2vec_4models.csv
💾 Saved ensemble_weights_rdkit_mol2vec_4models.csv


In [ ]:
# ─── Optional: Define custom weights for ensemble ───────────────────────────
# Uncomment and modify the weights below to use custom values instead of validation F1 weights

custom_weights = {
    "Extra Trees (RDKit)": 0.3,
    "XGBoost (RDKit)": 0.2,
    "SVM (RDKit)": 0.2,
    "XGBoost (Mol2Vec)": 0.3,
}

# Ensure weights sum to 1
weight_sum = sum(custom_weights.values())
if abs(weight_sum - 1.0) > 1e-6:
    print(f"⚠️ Warning: Weights sum to {weight_sum}, normalizing...")
    custom_weights = {k: v / weight_sum for k, v in custom_weights.items()}

# Recompute ensemble with custom weights
fit_ens_proba_cust = sum(custom_weights[name] * base_outputs[name]["fit_proba"] for name in base_outputs)
val_ens_proba_cust = sum(custom_weights[name] * base_outputs[name]["val_proba"] for name in base_outputs)
test_ens_proba_cust = sum(custom_weights[name] * base_outputs[name]["test_proba"] for name in base_outputs)

ensemble_outputs_cust = {
    "fit_pred": pred_from_proba(fit_ens_proba_cust),
    "val_pred": pred_from_proba(val_ens_proba_cust),
    "test_pred": pred_from_proba(test_ens_proba_cust),
}

# Evaluate custom ensemble
fit_metrics_ens_cust = compute_metrics(y_fit, ensemble_outputs_cust["fit_pred"])
val_metrics_ens_cust = compute_metrics(y_val, ensemble_outputs_cust["val_pred"])
test_metrics_ens_cust = compute_metrics(y_test, ensemble_outputs_cust["test_pred"])

cust_row = {
    "Model": "Ensemble(Custom Weights)",
    "Embedding": "Mixed",
    "Type": "Ensemble_Custom",
    "Ensemble_Weight": np.nan,
}
for k, v in fit_metrics_ens_cust.items():
    cust_row[f"Fit_{k}"] = round(v, 4)
for k, v in val_metrics_ens_cust.items():
    cust_row[f"Val_{k}"] = round(v, 4)
for k, v in test_metrics_ens_cust.items():
    cust_row[f"Test_{k}"] = round(v, 4)

results_df_cust = pd.concat([results_df, pd.DataFrame([cust_row])], ignore_index=True)
print("\n✅ Custom ensemble metrics computed:")
print(results_df_cust.loc[results_df_cust["Type"] == "Ensemble_Custom", :].iloc[0])

print("✅ Custom weights section ready (uncomment to use)")


✅ Custom ensemble metrics computed:
Model              Ensemble(Custom Weights)
Embedding                             Mixed
Type                        Ensemble_Custom
Fit_ACC                              0.9994
Fit_PRE                              0.9991
Fit_SPEC                             0.9998
Fit_SENS                             0.9991
Fit_F1                               0.9991
Fit_MCC                              0.9989
Val_ACC                              0.8817
Val_PRE                              0.8477
Val_SPEC                             0.9578
Val_SENS                             0.8233
Val_F1                               0.8334
Val_MCC                              0.7830
Test_ACC                             0.8893
Test_PRE                             0.8688
Test_SPEC                            0.9594
Test_SENS                            0.8415
Test_F1                              0.8533
Test_MCC                             0.7962
Ensemble_Weight                        